# 07 XGBoost on the G0–G3 feature sets

This notebook tunes and validates xgboost on the four cumulative feature sets built in
`02_feature_engineering.ipynb`: G0 raw columns (197) → G1 + row-level features (224) → G2 + card and device
history (234) → G3 + customer history (252). All six model notebooks (04 to 09) follow the same steps and call the
same code in `src/`, so their results can be compared directly in `10_model_comparison.ipynb`.

- **Data:** `data/processed/features_v2.parquet` (build it with `02_feature_engineering.ipynb`). `src.feature_sets`
  drops the test block before anything else sees it, so this notebook never touches it.
- **Splits:** `src/split.py`. Training block = the first 472,432 rows by time; validation = the next 59,054.
- **Every fit** appends one row to `results/log.csv`. Rerunning the notebook skips anything already logged, so it
  can be stopped and restarted. Validation scores are saved to `results/preds/` (not in git) for the comparison
  notebook and a possible ensemble.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# Import the model library before anything else: on macOS some model libraries clash with PyTorch's
# threading library if PyTorch is loaded first (see tests/conftest.py).
from src.models import xgb as model

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from src import data, experiment, feature_sets as fs

pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

## 1. Settings

- `FEATURE_SETS`: the sets to validate on.
- `TUNE_ON`: `"G3"` tunes once, on the full set, and validates those settings on every set, so the sets are
  compared with the model held fixed. `None` tunes every set separately (four times the tuning cost).
- `GRID`: the settings tried during tuning. The full grid is defined in the model's file in `src/models/`; every
  model uses its own grid as defined there. For a quick trial, use e.g. `model.GRID[:2]`, but log such a trial to a
  separate file (`LOG_PATH`) so it doesn't mix with the real runs.
- `SEEDS`: the validation seeds (0, 1, 2 for the group's results).
- `SAVE_OOF`: also save the time-CV fold scores of the chosen settings (4 extra fits per set), needed to train a
  stacking model later.

In [ ]:
FEATURE_SETS = ["G0", "G1", "G2", "G3"]
TUNE_ON = "G3"
GRID = model.GRID
SEEDS = [0, 1, 2]
SAVE_OOF = True
LOG_PATH = experiment.LOG_PATH          # results/log.csv
PREDS_DIR = experiment.PREDS_DIR        # results/preds/

print(f"{model.NAME}: {len(GRID)} settings to tune on {TUNE_ON or 'every set'}, seeds {SEEDS}")
print("defaults:", model.DEFAULTS)
pd.DataFrame(GRID)

## 2. Data

In [ ]:
df, train_idx, val_idx = fs.load_features()
spec = fs.load_spec()
sets = {name: fs.feature_list(name, spec) for name in FEATURE_SETS}
print(f"rows loaded: {len(df):,} (train {len(train_idx):,}, validation {len(val_idx):,}; test block not loaded)")
print({name: len(cols) for name, cols in sets.items()})

## 3. Model input

**Input:** the same `TreePipeline(kind="lgbm")` as LightGBM: missing values stay missing, categories stay pandas
categories with the training levels, and XGBoost splits on them natively (`enable_categorical=True`, histogram
method).

**Model:** `src/models/xgb.py`. Up to 1,500 trees with early stopping after 50 rounds without a better average
precision (the same measure as `src/metrics.py`), keeping the best round. "balanced" class weight means negatives /
positives of the training part. The settings grid is a fixed random sample of 20 combinations of depth, learning
rate, minimum child weight, column sampling and class weight.

In [ ]:
def make_pipeline(columns):
    return data.TreePipeline("lgbm", columns=columns)

## 4. Tune and validate

1. **Tune** with time CV inside the training block (seed 0): the block is cut into 30-day bins, and fold k trains on
   bins 0 to k−1 and scores bin k, so every fold scores months the model hasn't seen (4 folds). The settings with
   the highest mean average precision (PR-AUC) are chosen.
2. **Validate:** refit the chosen settings on the whole training block with each seed and score the validation
   block. The decision threshold (best F1) is picked on the validation block and stored in the log for the final
   test scoring.

Early stopping uses the rows being scored: the fold's validation part during tuning, and the validation
block in step 4. That slightly favours the boosting models and the MLP over logistic regression and random forest
in the validation numbers; the test block, scored once after the freeze, has no such advantage.

In [ ]:
chosen = experiment.tune_and_validate(model, make_pipeline, df, train_idx, val_idx, sets, GRID,
                                      tune_on=TUNE_ON, seeds=SEEDS, log_path=LOG_PATH,
                                      preds_dir=PREDS_DIR, save_oof=SAVE_OOF)
chosen

## 5. Results

All numbers below are read back from `results/log.csv`. Validation numbers are the mean and sample standard deviation
over the seeds, for the chosen settings only.

In [ ]:
cv, val = experiment.run_summary(model.NAME, FEATURE_SETS, LOG_PATH)
print("Tuning: mean time-CV PR-AUC over 4 folds, 5 best settings per tuned set")
display(cv.groupby("feature_set").head(5).round(4))

chosen_key = {name: json.dumps(p, sort_keys=True) for name, p in chosen.items()}
val = val[[chosen_key.get(r.feature_set) == r.settings for r in val.itertuples()]]
val = val.set_index("feature_set").reindex(FEATURE_SETS)
print("Validation block, chosen settings")
display(val.drop(columns="settings").round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(val.index, val["pr_auc_mean"], yerr=val["pr_auc_std"].fillna(0), color="#4C72B0", capsize=4)
for x, y in zip(val.index, val["pr_auc_mean"]):
    ax.text(x, y, f"{y:.3f}", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("validation PR-AUC (mean ± sd over seeds)")
ax.set_title(f"{model.NAME}: validation PR-AUC by feature set")
ax.grid(axis="x")
fig.tight_layout()
plt.show()

## 6. Findings

Write this after running, quoting only numbers from the tables above:

- Which feature set scores best, and how much each step adds (G0 → G1 → G2 → G3).
- Whether the chosen settings sit at the edge of the grid (a sign the grid should be widened).
- Anything unusual, such as a large spread between seeds.